In [83]:
# Load `.env` into the environment so keys like GEMINI_API_KEY are available to later cells.
import os
from dotenv import load_dotenv
load_dotenv()

True

In [84]:
# Sync GEMINI_API_KEY to GOOGLE_API_KEY; set model, max-output cap, and embedding throttle defaults (GEMINI_MAX_OUTPUT_TOKENS limits reply length).
import os

_gem = os.getenv("GEMINI_API_KEY")
if _gem:
    os.environ["GEMINI_API_KEY"] = _gem
    os.environ["GOOGLE_API_KEY"] = _gem

# Smaller model + low max tokens reduce free-tier quota pressure (override in .env).
os.environ.setdefault("GEMINI_MODEL", "gemini-2.5-flash-lite")
os.environ.setdefault("GEMINI_MAX_OUTPUT_TOKENS", "256")

# Embeddings free tier: ~100 embed_content requests/minute per model. Cap chunks and
# sleep between API calls (each LangChain batch can issue multiple requests).
os.environ.setdefault("GEMINI_EMBED_MAX_CHUNKS", "12")
os.environ.setdefault("GEMINI_EMBED_API_PAUSE_SEC", "1.05")
os.environ.setdefault("GEMINI_EMBED_ON_429_WAIT", "48")

## Langsmith Tracking and tracing

# os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_TRACING_V2"]="true"
# os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

'48'

In [85]:
# Construct the Gemini chat model; `max_output_tokens` comes from GEMINI_MAX_OUTPUT_TOKENS (raise it if answers cut off early).
from langchain_google_genai import ChatGoogleGenerativeAI

_model = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash-lite")
_max_out = int(os.environ.get("GEMINI_MAX_OUTPUT_TOKENS", "256"))
llm = ChatGoogleGenerativeAI(model=_model, max_output_tokens=_max_out)
print(llm)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


output_version=None profile={'name': 'Gemini 2.5 Flash Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') location=None model='gemini-2.5-flash-lite' max_output_tokens=256 client=<google.genai.client.Client object at 0x000002CB1C591670> default_metadata=() model_kwargs={}


In [86]:
# Smoke-test the LLM with a short prompt (still subject to max_output_tokens cap).
result = llm.invoke("Define agentic AI in one sentence.")
print(result)

content='Agentic AI refers to artificial intelligence systems that can autonomously perceive their environment, make decisions, and take actions to achieve specific goals.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e27f8-ad12-7450-a5b5-7f8cfe4313b6-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 9, 'output_tokens': 26, 'total_tokens': 35, 'input_token_details': {'cache_read': 0}}


In [87]:
# Show only the assistant string from the last `llm.invoke` result.
print(result.content)

Agentic AI refers to artificial intelligence systems that can autonomously perceive their environment, make decisions, and take actions to achieve specific goals.


In [88]:
# Define a chat prompt template (system + user) reused by chains below.
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","Answer briefly in a few sentences."),
        ("user","{input}")

    ]
)
prompt




ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Answer briefly in a few sentences.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [49]:
# Rebuild the LLM if you skipped earlier cells (same GEMINI_MAX_OUTPUT_TOKENS ceiling applies).
from langchain_google_genai import ChatGoogleGenerativeAI

_model = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash-lite")
_max_out = int(os.environ.get("GEMINI_MAX_OUTPUT_TOKENS", "256"))
llm = ChatGoogleGenerativeAI(model=_model, max_output_tokens=_max_out)

In [50]:
# LCEL chain: prompt | llm (each completion still capped by max_output_tokens on the LLM).
chain=prompt|llm 

response=chain.invoke({"input": "What is LangSmith? (2 sentences max)"})
print(response)

content='LangSmith is a development platform designed to help developers build, test, and deploy applications powered by large language models (LLMs). It provides tools for tracing, debugging, and evaluating LLM-based applications, enabling users to understand and improve their performance.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e27e6-2801-7982-bf7c-9cb866eec4fc-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 18, 'output_tokens': 51, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}


In [51]:
# Print the chain’s assistant text only.
print(response.content)

LangSmith is a development platform designed to help developers build, test, and deploy applications powered by large language models (LLMs). It provides tools for tracing, debugging, and evaluating LLM-based applications, enabling users to understand and improve their performance.


In [52]:
# Same chain ending in StrOutputParser so `invoke` returns a plain string (generation cap unchanged).
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()

chain=prompt|llm|output_parser

response=chain.invoke({"input": "What is LangSmith? (2 sentences max)"})
print(response)

LangSmith is a development platform designed to help developers build, test, and monitor applications powered by large language models (LLMs). It provides tools for tracing, debugging, and evaluating LLM interactions, making it easier to manage the complexities of LLM-based applications.


In [53]:
# Prompt tuned for JSON output via JsonOutputParser (model must finish within max_output_tokens).
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser


output_parser=JsonOutputParser()
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)


In [54]:
# Run prompt | llm | JsonOutputParser and inspect structured output (watch truncation if max_output is low).
chain=prompt|llm|output_parser

response=chain.invoke({"query": "What is LangSmith? (2 sentences max)"})
print(response)

{'answer': 'LangSmith is a platform for developing, testing, and monitoring large language model (LLM) applications. It provides tools for tracing LLM calls, evaluating performance, and debugging complex workflows, enabling developers to build more robust and reliable AI-powered applications.'}


*This section walks through a small RAG pipeline in ordered steps.*

### RAG pipeline (steps)

1. **Load** — fetch HTML and build `Document` objects.  
2. **Chunking** — split long text into fixed-size segments (`chunk_size`).  
3. **Overlapping** — reuse characters at chunk edges (`chunk_overlap`) so ideas are not split cleanly in half.  
4. **Embed** — map each chunk to a vector with Gemini embeddings.  
5. **Index** — store vectors in FAISS for fast similarity search.  
6. **Retrieve** — embed the user query and pull the closest chunks.  
7. **Generate** — stuff retrieved text into a prompt and call the LLM.


In [55]:
# Step 1 (load) — import the HTML loader used to fetch pages into LangChain `Document` objects.
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [56]:
# Step 1 (load) — choose the tutorial URL you will download and index.
loader=WebBaseLoader("https://python.langchain.com/docs/tutorials/llm_chain/")
loader

In [57]:
# Step 1 (load) — run the loader and return raw documents (one long page before chunking / overlap).
document=loader.load()
document

[Document(metadata={'source': 'https://python.langchain.com/docs/tutorials/llm_chain/', 'title': 'Build a RAG agent with LangChain - Docs by LangChain', 'language': 'en'}, page_content='Build a RAG agent with LangChain - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChainBuild a RAG agent with LangChainDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainSemantic searchRAG agentSQL agentVoice agentMulti-agentLangGraphConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesLangChain AcademyCase studiesGet helpOn this pageOverviewConceptsPreviewSetupInstallationLangSmithComponents1. IndexingLoading documentsSplitting documentsStoring documents2. Retr

In [71]:
# Steps 2–3 — chunking + overlapping: split big text, then keep a shared tail/head window between neighbors.
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

_max = int(os.environ.get("GEMINI_EMBED_MAX_CHUNKS", "12"))
# Step 2 — chunking: `chunk_size` caps characters per segment (smaller => more chunks / more embed API calls).
# Step 3 — overlapping: `chunk_overlap` duplicates that many chars across consecutive chunks so splits do not drop context.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=4500, chunk_overlap=300)
documents = text_splitter.split_documents(document)
documents = documents[:_max]  # still cap total chunks for Gemini free-tier embed RPM
print(f"Indexing {len(documents)} chunks (GEMINI_EMBED_MAX_CHUNKS={_max})")
documents


Indexing 8 chunks (GEMINI_EMBED_MAX_CHUNKS=12)


[Document(metadata={'source': 'https://python.langchain.com/docs/tutorials/llm_chain/', 'title': 'Build a RAG agent with LangChain - Docs by LangChain', 'language': 'en'}, page_content='Build a RAG agent with LangChain - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChainBuild a RAG agent with LangChainDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainSemantic searchRAG agentSQL agentVoice agentMulti-agentLangGraphConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesLangChain AcademyCase studiesGet helpOn this pageOverviewConceptsPreviewSetupInstallationLangSmithComponents1. IndexingLoading documentsSplitting documentsStoring documents2. Retr

In [72]:
# Step 4 — embeddings: turn each chunk into a vector (Gemini; throttled via GEMINI_EMBED_*; separate from chat max_output).
import os
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai._common import GoogleGenerativeAIError


class _RateLimitedGeminiEmbeddings(GoogleGenerativeAIEmbeddings):
    """One document per embed API round-trip, plus pause, so inner batching cannot burst past RPM."""

    def embed_documents(self, texts, *, batch_size: int = 100, **kwargs):
        if not texts:
            return []
        pause = float(os.environ.get("GEMINI_EMBED_API_PAUSE_SEC", "1.05"))
        on_429 = float(os.environ.get("GEMINI_EMBED_ON_429_WAIT", "48"))
        vectors: list[list[float]] = []
        for j, text in enumerate(texts):
            for attempt in range(3):
                try:
                    vectors.extend(
                        super().embed_documents([text], batch_size=1, **kwargs)
                    )
                    break
                except GoogleGenerativeAIError as err:
                    msg = str(err)
                    if attempt < 2 and (
                        "429" in msg or "RESOURCE_EXHAUSTED" in msg
                    ):
                        time.sleep(on_429)
                        continue
                    raise
            if j + 1 < len(texts) and pause > 0:
                time.sleep(pause)
        return vectors


_emb_model = os.environ.get("GEMINI_EMBEDDING_MODEL", "models/gemini-embedding-001")
embeddings = _RateLimitedGeminiEmbeddings(model=_emb_model)


In [77]:
# Step 5 — indexing: add chunk vectors to FAISS (`faiss-cpu`) so you can search by semantic similarity.
from langchain_community.vectorstores import FAISS

vectorstore=FAISS.from_documents(documents,embeddings)
vectorstore

In [78]:
# Step 6 — retrieval: embed the question and fetch the nearest indexed chunks (similarity_search).
query="This is a relatively simple LLM application "

result=vectorstore.similarity_search(query)
result[0].page_content

'Go deeper\nEmbeddings: Wrapper around a text embedding model, used for converting text to embeddings.\n\nIntegrations: 30+ integrations to choose from.\nInterface: API reference for the base interface.\n\nVectorStore: Wrapper around a vector database, used for storing and querying embeddings.\n\nIntegrations: 40+ integrations to choose from.\nInterface: API reference for the base interface.\n\nThis completes the Indexing portion of the pipeline. At this point we have a query-able vector store containing the chunked contents of our blog post. Given a user question, we should ideally be able to return the snippets of the blog post that answer the question.\n\u200b2. Retrieval and generation\nRAG applications commonly work as follows:\n\nRetrieve: Given a user input, relevant splits are retrieved from storage using a Retriever.\nGenerate: A model produces an answer using a prompt that includes both the question with the retrieved data\n\n\nNow let’s write the actual application logic. We

In [80]:
# Step 7a — generation prompt: inject retrieved passages as `{context}` (answer still bounded by LLM max_output_tokens).
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

In [82]:
# Step 7b — generation chain: combine retrieved docs + LLM (`create_stuff_documents_chain` “stuffs” all context into one prompt).
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool